# 04 — Pipeline TTS

ElevenLabs, OpenAI, Cartesia, LMNT, Rime.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises TTS provider construction and (T3) live synthesis.


### TTS provider construction


In [ ]:
import { ElevenLabsTTS, OpenAITTS, CartesiaTTS, RimeTTS, LMNTTTS } from "getpatter";
await cell('tts_providers', { tier: 1, env }, () => {
  const el = new ElevenLabsTTS({ apiKey: 'el-test', voiceId: '21m00Tcm4TlvDq8ikWAM' });
  const ot = new OpenAITTS({ apiKey: 'sk-test', voice: 'alloy', model: 'tts-1' });
  const ca = new CartesiaTTS({ apiKey: 'ca-test' });
  const ri = new RimeTTS({ apiKey: 'ri-test' });
  const lm = new LMNTTTS({ apiKey: 'lm-test' });
  for (const [name, p] of [['ElevenLabs', el], ['OpenAI', ot], ['Cartesia', ca], ['Rime', ri], ['LMNT', lm]]) {
    console.log(`${name}: ${(p as any).constructor.name}`);
  }
});


### TTS text preparation


In [ ]:
import { filterForTts } from "getpatter";
await cell('tts_text_prep', { tier: 1, env }, () => {
  const samples = ['**Bold** text.', 'Visit https://example.com 🎉', 'Code: `x = 1`'];
  for (const raw of samples) {
    console.log(`in:  ${raw}`);
    console.log(`out: ${filterForTts(raw)}`);
  }
});


### Live: ElevenLabs TTS synthesis  *(T3 — requires `ELEVENLABS_API_KEY`)*


In [ ]:
await cell('elevenlabs_tts_live', { tier: 3, required: ['elevenLabsKey'], env }, async () => {
  const voiceId = '21m00Tcm4TlvDq8ikWAM';
  const resp = await fetch(`https://api.elevenlabs.io/v1/text-to-speech/${voiceId}`, {
    method: 'POST',
    headers: { 'xi-api-key': env.elevenLabsKey, 'Content-Type': 'application/json' },
    body: JSON.stringify({ text: 'Hello from Patter.', model_id: 'eleven_monolingual_v1' }),
  });
  const buf = await resp.arrayBuffer();
  console.log(`ElevenLabs audio: ${buf.byteLength} bytes`);
});


### Live: OpenAI TTS synthesis  *(T3 — requires `OPENAI_API_KEY`)*


In [ ]:
await cell('openai_tts_live', { tier: 3, required: ['openaiKey'], env }, async () => {
  const resp = await fetch('https://api.openai.com/v1/audio/speech', {
    method: 'POST',
    headers: { Authorization: `Bearer ${env.openaiKey}`, 'Content-Type': 'application/json' },
    body: JSON.stringify({ model: 'tts-1', voice: 'alloy', input: 'Hello from Patter.' }),
  });
  const buf = await resp.arrayBuffer();
  console.log(`OpenAI TTS audio: ${buf.byteLength} bytes`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.
